> **Save your own copy first.** Unless this notebook is already in your own Google Drive, Colab does not keep your changes. Go to **File → Save a copy in Drive** and work in the copy that opens in a new tab. You find it later in Google Drive, in *Colab Notebooks*.

![Cultural Data Analysis · Week 3: Structured Data and Machine Learning](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week3-banner.png)

# Data critique: from a table to a label

This notebook goes with the ten questions of the [data critique](https://goto4711.github.io/cda-teaching-tools/workshops/week3/#critique). In the workshop, it loads the donation records of *Global Aid Alliance*; the case and its data sheet are on the workshop page. Most questions are answered by reading the data sheet, looking at the rows and discussing.

Write your answers in the [data critique template](https://github.com/goto4711/cda-teaching-tools/blob/main/notebooks/templates/data_critique_template.md), a Markdown file you keep for your group project (download it with the download button on that page). Use this notebook to test what you write: when an answer says *we saw in the data*, show it here.

The cells below help with the questions that matter once a machine learns from the table: what would it predict (the **label**), whom does that label favour, where do the **features** come from, and can a harmless-looking column stand in for a sensitive one?

**In your group project**, use it on your own data: change `DATA_FILE` below, and the settings at the top of each cell.

Run the cell below to load the data.

In [ ]:
#Keep cell
import pandas as pd

# A web address, or a file you uploaded to Colab (folder symbol on the left), e.g. 'my_data.csv'
DATA_FILE = 'https://raw.githubusercontent.com/goto4711/cdai/main/donor_records_export.csv'

if DATA_FILE.endswith(('.xlsx', '.xls')):
  data = pd.read_excel(DATA_FILE)
else:
  data = pd.read_csv(DATA_FILE)       # for a file separated by semicolons: pd.read_csv(DATA_FILE, sep=';')
print(len(data), 'rows,', len(data.columns), 'columns')
data.head()

---

## 1. Make the label (question 5)

The NGO wants to keep donors giving. A model that predicts which donors stay needs a column that says, for every donor, whether they stayed: the **label**. The data has no such column. Someone has to define it, and every definition is a **proxy** for "loyalty".

The next cell makes three labels, one row per donor:

* **A:** gave in at least two different years;
* **B:** gave again within 12 months of an earlier gift;
* **C:** at least one gift is marked as a recurring gift (`recurring_gift`).

Change the definitions, or add your own. Note that a donor is whatever the column `donor_id` stands for: compare the column `names` (how many different names each ID has).

Run the next cell.

In [ ]:
ID, DATE = 'donor_id', 'donation_date'

dates = pd.to_datetime(data[DATE], errors='coerce')
donors = data.assign(date=dates, year=dates.dt.year).sort_values('date').groupby(ID).agg(
  gifts=('date', 'size'),
  years=('year', 'nunique'),
  names=('donor_name', 'nunique'),
  again_within_12_months=('date', lambda d: bool((d.diff().dt.days <= 365).any())),
  recurring=('recurring_gift', lambda r: bool(r.str.strip().str.lower().str.startswith('yes').any())),
)
donors['loyal_A'] = donors['years'] >= 2
donors['loyal_B'] = donors['again_within_12_months']
donors['loyal_C'] = donors['recurring']
LABELS = ['loyal_A', 'loyal_B', 'loyal_C']

print(len(donors), 'donors. Share counted as loyal:')
print(donors[LABELS].mean().round(2).to_string())
print('\nDo the labels agree? (A against B)')
print(pd.crosstab(donors['loyal_A'], donors['loyal_B']))
donors.head()

How many donors are "loyal" depends on the definition. Which donors change sides? What does each definition count, and what does it miss: a donor who gives a large gift every two years, a donor who gave once and died, two people sharing one ID?

---

## 2. Whom does the label favour? (questions 5, 9)

A model learns to find donors like the ones its label calls loyal. Who are they? The next cell gives the share of loyal donors per group, for each label. Each donor gets the value they have most often in `GROUP`. Try `'channel'`, `'gender'`, `'age_range'` and `'region'`.

Run the next cell.

In [ ]:
GROUP = 'channel'

per_donor = data.groupby(ID)[GROUP].agg(lambda s: s.mode().iat[0] if s.notna().any() else '(missing)')
table = donors[LABELS].groupby(per_donor).mean().round(2)
table['donors'] = per_donor.value_counts()
table

Which groups would the fundraising target, and which would it give up on? Is a difference real, or a trace of how the data was made, e.g. of how gifts by post are recorded? Small groups give unstable shares: look at the column `donors`.

---

## 3. Where do the features come from? (question 6)

The other columns become the **features** the model learns from. Who filled them in: the donors themselves on a website, staff from a letter, a volunteer at an event? If that differs by channel, the model may learn the channel rather than the donor. The next cell shows the share of each value of `COLUMN` per value of `SOURCE`, with missing values as `(missing)`.

Run the next cell.

In [ ]:
SOURCE, COLUMN = 'channel', 'gender'

pd.crosstab(data[SOURCE].fillna('(missing)'), data[COLUMN].fillna('(missing)'), normalize='index').round(2)

---

## 4. Proxies (questions 8, 9)

Leaving a sensitive column out does not keep it out of the model if another column gives it away: in real data, a postcode can stand for income or ethnicity, a first name for gender, a campaign for religion. The next cell shows how well `FEATURE` predicts `SENSITIVE`: for each value of `FEATURE`, the share of each value of `SENSITIVE`. The more these rows differ from one another, the more the feature gives away.

Run the next cell.

In [ ]:
FEATURE, SENSITIVE = 'campaign', 'age_range'

pd.crosstab(data[FEATURE].fillna('(missing)'), data[SENSITIVE].fillna('(missing)'), normalize='index').round(2)

In this made-up data the columns are hardly related. In your own data, think first about which columns *could* stand in for a sensitive one, and check those.

**Your own checks**